<a href="https://colab.research.google.com/github/tolumeadows/ai-portrait-pipeline/blob/main/Diffuser_stuff_idk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from diffusers import AutoPipelineForText2Image # allows us to easily do text to image stuff
from diffusers.utils import load_image # allows us to load images with urls
import torch # needed for specifying numeric precision later on

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    safety_checker=None,
    requires_safety_checker=False,
    torch_dtype=torch.float16).to("cuda") # this inputs the diffusion model from hugging face we want, runnning it on 16 bit acc for quickness and on the gpu we have.

In [ ]:
pipeline.load_ip_adapter( # this adds on to our pipeline a code that allows us to input images to generate images, in combination with our baseline text-to-image
    "h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-full-face_sd15.bin"
)

In [ ]:
# image = load_image("/content/alina-chernovolova-CupJvkGjGGE-unsplash.jpg")
# image = load_image("/content/alina-chernovolova-KwdnBzW8ixA-unsplash.jpg")
image = load_image("/christian-agbede-_PfE8dZSPUs-unsplash.jpg")
image

In [ ]:
pipeline.set_ip_adapter_scale(.3) # sets the strengths of the inputed image to 0.8 and not the full 1. or to whatever if i change it.

In [ ]:
# for seed_img in range(1,3):
#   output = pipeline( # runs with the pipeline installs above
#       prompt="a zoomed out full body portrait in medieval armor, renaissance painting style. the background is a desert landscape.", # prompt
#       ip_adapter_image=image, # takes in the image we set to "image" above
#       negative_prompt="low quality, blurry", # negative prompt the pipeline should avoid
#       num_inference_steps=50, # the number of steps of denoising to get to the image
#       height=640,
#       width=512,
#       generator=torch.Generator("cuda").manual_seed(seed_img) # manually sets the random starting static to the same each time since the starting noise/static is generated by a "random" number generator
#   ).images[0] # this pipline would return a list of images, but we take just the first one

#   final= output.resize((750,1000))
#   display(final)

In [ ]:
# I LIKE THIS MORE RIGHT NOW. MY OWN!

# output = pipeline( # runs with the pipeline installs above
#     prompt="a zoomed out full body portrait in medieval armor, renaissance painting style. the background is a desert landscape.", # prompt
#     ip_adapter_image=image, # takes in the image we set to "image" above
#     negative_prompt="low quality, blurry", # negative prompt the pipeline should avoid
#     num_inference_steps=50, # the number of steps of denoising to get to the image
#     height=640,
#     width=512,
#     generator=torch.Generator("cuda").manual_seed(42) # manually sets the random starting static to the same each time since the starting noise/static is generated by a "random" number generator
# ).images[0] # this pipline would return a list of images, but we take just the first one

# final= output.resize((750,1000))
# display(final)

# WHAT CLAUDE SAID WAS BETTER.

pipeline.set_ip_adapter_scale(0.7)

for seed in range(1, 5):
    output = pipeline(
        prompt="a waist-up portrait in medieval armor, renaissance painting style, desert landscape background",
        ip_adapter_image=image,
        negative_prompt="low quality, blurry",
        num_inference_steps=30,
        height=640,
        width=512,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    print("seed", seed)
    display(output)

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image, DDIMScheduler
from transformers import CLIPVisionModelWithProjection

image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter",
    subfolder="models/image_encoder",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

pipeline = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    image_encoder=image_encoder,
    torch_dtype=torch.float32,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)

pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter-plus-face_sd15.safetensors",
)
pipeline.set_ip_adapter_scale(0.6)